In [68]:
import pandas as pd
from pathlib import Path
import sys
import numpy as np
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt
import prince
from itertools import combinations
import plotly.express as px

In [48]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw"
titanic_raw = pd.read_csv(DATA_PATH / "Titanic-Dataset.csv")

titanic_raw.dtypes.to_frame("dtype")

,dtype
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [ ]:
# Mudança de variaveis para qualitativa e a criação de uma nova coluna com os grupos de pessoas
# Antes for feito uma divisão igualitaria das idades, porém o p-value deu 88%.

titanic_raw['Survived'] = titanic_raw['Survived'].map({0: 'Dead',1: 'Survived'})
titanic_raw['Pclass'] = titanic_raw['Pclass'].map({1: 'First Class',2: 'Second Class',3:'Third Class'})
titanic_raw['Age_cat'] = pd.cut(titanic_raw['Age'],bins=[0, 12, 18, 60, 100],labels=['Child', 'Teen', 'Adult', 'Elder'])


In [ ]:
# Tabela de frequencia

tabela_mca_1 = pd.crosstab(titanic_raw["Survived"], titanic_raw["Sex"])
tabela_mca_2 = pd.crosstab(titanic_raw["Survived"], titanic_raw["Age_cat"])
tabela_mca_3 = pd.crosstab(titanic_raw["Survived"], titanic_raw["Pclass"])

print(tabela_mca_1)
print(tabela_mca_2)
print(tabela_mca_3)

Sex       female  male
Survived              
Dead          81   468
Survived     233   109
Age_cat   Child  Teen  Adult  Elder
Survived                           
Dead         29    40    338     17
Survived     40    30    215      5
Pclass    First Class  Second Class  Third Class
Survived                                        
Dead               80            97          372
Survived          136            87          119


In [ ]:
# Calculo da estatistica qui quadrado e p-value

tab_1 = chi2_contingency(tabela_mca_1)

print("Survived x Sex")
print(f"estatística qui²: {round(tab_1[0], 2)}")
print(f"p-valor da estatística: {round(tab_1[1], 4)}")
print(f"graus de liberdade: {tab_1[2]}")

tab_2 = chi2_contingency(tabela_mca_2)

print("Survived x Age_cat")
print(f"estatística qui²: {round(tab_2[0], 2)}")
print(f"p-valor da estatística: {round(tab_2[1], 4)}")
print(f"graus de liberdade: {tab_2[2]}")

tab_3 = chi2_contingency(tabela_mca_3)

print("Survived x Pclass")
print(f"estatística qui²: {round(tab_3[0], 2)}")
print(f"p-valor da estatística: {round(tab_3[1], 4)}")
print(f"graus de liberdade: {tab_3[2]}")

Survived x Sex
estatística qui²: 260.72
p-valor da estatística: 0.0
graus de liberdade: 1
Survived x Age_cat
estatística qui²: 12.37
p-valor da estatística: 0.0062
graus de liberdade: 3
Survived x Pclass
estatística qui²: 102.89
p-valor da estatística: 0.0
graus de liberdade: 2


In [61]:
titanic_mca = titanic_raw.drop(columns=['PassengerId','Name','Age','Parch','Ticket','Fare','Cabin','Embarked','SibSp'])
mca = prince.MCA(n_components=3).fit(titanic_mca)

In [62]:
# Quantidade de dimensões = qtde total de categorias - qtde de variáveis

# Quantidade total de categorias
mca.J_

# Quantidade de variáveis na análise
mca.K_

# Quantidade de dimensões
quant_dim = mca.J_ - mca.K_

# Resumo das informações
print(f"quantidade total de categorias: {mca.J_}")
print(f"quantidade de variáveis: {mca.K_}")
print(f"quantidade de dimensões: {quant_dim}")

quantidade total de categorias: 11
quantidade de variáveis: 4
quantidade de dimensões: 7


In [63]:
tabela_autovalores = mca.eigenvalues_summary

print(tabela_autovalores)

# Soma de todos os autovalores (todas as dimensões existentes)

print(mca.total_inertia_)

          eigenvalue % of variance % of variance (cumulative)
component                                                    
0              0.454        24.50%                     24.50%
1              0.308        16.63%                     41.12%
2              0.271        14.63%                     55.76%
1.8532571013317747


In [64]:
# É interessante plotar apenas dimensões com autovalores maiores do que a média

print(mca.total_inertia_/quant_dim)

# Neste caso, as 3 dimensões extraídas têm autovalores > 0.199

0.2647510144759678


In [65]:
# Obtendo as coordenadas principais das categorias das variáveis

coord_burt = mca.column_coordinates(titanic_mca)

print(coord_burt)

                             0         1         2
Survived__Dead       -0.716295  0.027725 -0.005254
Survived__Survived    1.100280 -0.066923  0.054871
Pclass__First Class   0.738559  1.195163  0.485407
Pclass__Second Class  0.357704 -0.146233 -1.467945
Pclass__Third Class  -0.493475 -0.486588  0.368911
Sex__female           1.061021 -0.362246  0.173742
Sex__male            -0.606778  0.183846 -0.067026
Age_cat__Child        0.356751 -1.728279 -0.031383
Age_cat__Teen         0.216679 -1.074402  1.785716
Age_cat__Adult        0.033476  0.265642 -0.386587
Age_cat__Elder       -0.338405  3.207198  1.968387


In [66]:
# Obtendo as coordenadas-padrão das categorias das variáveis

coord_padrao = mca.column_coordinates(titanic_mca)/np.sqrt(mca.eigenvalues_)

print(coord_padrao)

                             0         1         2
Survived__Dead       -1.063047  0.049948 -0.010090
Survived__Survived    1.632916 -0.120563  0.105368
Pclass__First Class   1.096089  2.153112  0.932127
Pclass__Second Class  0.530865 -0.263441 -2.818895
Pclass__Third Class  -0.732362 -0.876599  0.708419
Sex__female           1.574651 -0.652594  0.333638
Sex__male            -0.900513  0.331202 -0.128710
Age_cat__Child        0.529451 -3.113532 -0.060264
Age_cat__Teen         0.321571 -1.935557  3.429112
Age_cat__Adult        0.049681  0.478560 -0.742363
Age_cat__Elder       -0.502224  5.777835  3.779895


In [67]:
# Obtendo as coordenadas das observações do banco de dados

# Na função, as coordenadas das observações vêm das coordenadas-padrão

coord_obs = mca.row_coordinates(titanic_mca)

print(coord_obs)

            0         1         2
0   -0.661560 -0.004222 -0.043186
1    1.088334  0.464629  0.157192
2    0.631222 -0.292799  0.101265
3    1.088334  0.464629  0.157192
4   -0.661560 -0.004222 -0.043186
..        ...       ...       ...
886 -0.345754  0.149067 -0.925015
887  1.088334  0.464629  0.157192
888 -0.073586 -0.493082  0.343989
889  0.469543  0.710578  0.041605
890 -0.661560 -0.004222 -0.043186

[891 rows x 3 columns]


In [ ]:
# Plotando o mapa perceptual (coordenadas-padrão)

chart = coord_padrao.reset_index()

var_chart = pd.Series(chart['index'].str.split('_', expand=True).iloc[:,0])

nome_categ=[]
for col in titanic_mca:
    nome_categ.append(titanic_mca[col].sort_values(ascending=True).unique())
    categorias = pd.DataFrame(nome_categ).stack().reset_index()

chart_df_mca = pd.DataFrame({'categoria': chart['index'],
                             'obs_x': chart[0],
                             'obs_y': chart[1],
                             'obs_z': chart[2],
                             'variavel': var_chart,
                             'categoria_id': categorias[0]})

# Segundo passo: gerar o gráfico de pontos

fig = px.scatter_3d(chart_df_mca, 
                    x='obs_x', 
                    y='obs_y', 
                    z='obs_z',
                    color='variavel',
                    text=chart_df_mca.categoria_id)

fig.write_html('assoc_mca_adapta.html')